# Fake News Classifier - Model Training

This notebook documents the training process of the Fake News Classifier using the ISOT Fake News Dataset.

**Machine Learning Pipeline:**
1. Load Fake and Real news datasets
2. Label the data
3. Combine headline and article text
4. Split data into training and testing sets
5. Convert text into TF-IDF features
6. Train a Logistic Regression classifier
7. Evaluate the model
8. Save the trained model and vectorizer

In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


## 1. Load the Dataset

The dataset contains separate CSV files for fake and real news articles.

In [2]:
fake = pd.read_csv("dataset/Fake.csv")
true = pd.read_csv("dataset/True.csv")

print("Fake news articles:", len(fake))
print("Real news articles:", len(true))

Fake news articles: 23481
Real news articles: 21417


## 2. Label the Data

Fake news is assigned label `0`, while real news is assigned label `1`.

In [3]:
fake["label"] = 0
true["label"] = 1

data = pd.concat([fake, true], ignore_index=True)

print("Total articles:", len(data))

Total articles: 44898


## 3. Prepare the Text Data

The headline and article body are combined into one text field.

In [4]:
data["text"] = data["title"].fillna("") + " " + data["text"].fillna("")

data = data[["text", "label"]]
data = data.dropna()
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

data.head()

,text,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,0
1,Trump drops Steve Bannon from National Securit...,1
2,Puerto Rico expects U.S. to lift Jones Act shi...,1
3,OOPS: Trump Just Accidentally Confirmed He Le...,0
4,Donald Trump heads for Scotland to reopen a go...,1


## 4. Split the Dataset

80% of the data is used for training and 20% is used for testing.

In [5]:
X = data["text"]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 35918
Testing samples: 8980


## 5. TF-IDF Feature Extraction

TF-IDF converts the news text into numerical features that can be processed by the machine learning model.

In [6]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=100000
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF training shape:", X_train_tfidf.shape)
print("TF-IDF testing shape:", X_test_tfidf.shape)

TF-IDF training shape: (35918, 100000)
TF-IDF testing shape: (8980, 100000)


## 6. Train the Logistic Regression Model

In [7]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

print("Model training completed!")

Model training completed!


## 7. Evaluate the Model

In [8]:
predictions = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, predictions)

print(f"Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(
    y_test,
    predictions,
    target_names=["Fake", "Real"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))

Accuracy: 98.65%

Classification Report:
              precision    recall  f1-score   support

        Fake       0.99      0.98      0.99      4696
        Real       0.98      0.99      0.99      4284

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980

Confusion Matrix:
[[4621   75]
 [  46 4238]]


## 8. Save the Trained Model

The trained Logistic Regression model and TF-IDF vectorizer are saved as `.pkl` files. These files are later loaded by the Flask web application.

In [9]:
joblib.dump(model, "fake_news_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("Model saved successfully!")
print("fake_news_model.pkl")
print("tfidf_vectorizer.pkl")

Model saved successfully!
fake_news_model.pkl
tfidf_vectorizer.pkl


# Conclusion

The Fake News Classifier was trained using TF-IDF feature extraction and Logistic Regression. The resulting model is integrated into a Flask web application where users can enter a headline and article text for classification.